<a href="https://colab.research.google.com/github/elhamod/IS834_Fall_2026/blob/main/Session%2007%20-%20Practice%20Lab%20-%20End-to-End%20Analysis/07_Practice_Lab_Solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IS834 — Session 7: Practice Lab (Solutions)

These are the worked solutions to the Session 7 practice lab. Every exercise has code and a written answer, and the outputs shown come from running the notebook top to bottom.

Try each exercise in the [exercise notebook](https://colab.research.google.com/github/elhamod/IS834_Fall_2026/blob/main/Session%2007%20-%20Practice%20Lab%20-%20End-to-End%20Analysis/07_Practice_Lab.ipynb) before you read its solution here. There is usually more than one correct way to write the code; what matters is that yours answers the question and that you can explain why.

Both datasets are read straight from the TidyTuesday repository on GitHub, so there is nothing to upload or install.

*Written for IS834 Fall 2026 by Mohannad Elhamod. Datasets from the [TidyTuesday](https://github.com/rfordatascience/tidytuesday) project (sources at the end of the notebook).*

## How this session works

There are seven exercises on two datasets, each about 20 minutes of work. In class you will get time to try each exercise on your own first. Then we will work through some of them together, and the rest are left as homework.

| Part | Dataset | Exercises | Time |
| --- | --- | --- | --- |
| A | Chocolate bar ratings (2,530 reviews) | 1–3 | ~60 min |
| — | Break | | ~15 min |
| B | Hotel bookings (119,390 bookings) | 4–7 | ~80 min |

# Setup

In [1]:
# The only package this lab needs. Colab already has it installed.
import pandas as pd

# Part A — Chocolate bar ratings

This dataset holds expert reviews of dark chocolate bars from the [Flavors of Cacao](http://flavorsofcacao.com/chocolate_database.html) database. **Goal:** decide which bars a specialty food retailer should stock.

Each row is one bar, reviewed once. The columns you will use most:

| Column | Meaning |
| --- | --- |
| `company_manufacturer` | Who made the bar |
| `company_location` | Country the maker is based in |
| `review_date` | Year the review was published |
| `country_of_bean_origin` | Country the cocoa beans came from |
| `cocoa_percent` | Cocoa content of the bar |
| `ingredients` | Number of ingredients, then their codes: B = beans, S = sugar, S* = other sweetener, C = cocoa butter, V = vanilla, L = lecithin, Sa = salt |
| `rating` | Expert rating, from 1 (unpleasant) to 4 (outstanding) |

In [2]:
# Load the chocolate reviews straight from the TidyTuesday repository on GitHub.
choc = pd.read_csv("https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2022/2022-01-18/chocolate.csv")
choc.head()

,ref,company_manufacturer,company_location,review_date,country_of_bean_origin,specific_bean_origin_or_bar_name,cocoa_percent,ingredients,most_memorable_characteristics,rating
0,2454,5150,U.S.A.,2019,Tanzania,"Kokoa Kamili, batch 1",76%,"3- B,S,C","rich cocoa, fatty, bready",3.25
1,2458,5150,U.S.A.,2019,Dominican Republic,"Zorzal, batch 1",76%,"3- B,S,C","cocoa, vegetal, savory",3.50
2,2454,5150,U.S.A.,2019,Madagascar,"Bejofo Estate, batch 1",76%,"3- B,S,C","cocoa, blackberry, full body",3.75
3,2542,5150,U.S.A.,2021,Fiji,"Matasawalevu, batch 1",68%,"3- B,S,C","chewy, off, rubbery",3.00
4,2546,5150,U.S.A.,2021,Venezuela,"Sur del Lago, batch 1",72%,"3- B,S,C","fatty, earthy, moss, nutty,chalky",3.00


## Exercise 1 — What are we working with? *(~20 min)*

**Exercise 1.** Describe what the data covers.

1. How many bars, makers and bean-origin countries are there, and which years do the reviews span?
2. What share of the bars come from each of the five most common bean-origin countries?
3. Which column, if any, could serve as an ID for each bar?

In [3]:
# Size of the table, and how many distinct makers and origins it covers.
print(choc.shape)
print("Makers:", choc["company_manufacturer"].nunique())
print("Bean origins:", choc["country_of_bean_origin"].nunique())
print("Reviews from", choc["review_date"].min(), "to", choc["review_date"].max())

(2530, 10)
Makers: 580
Bean origins: 62
Reviews from 2006 to 2021


In [4]:
# Share of bars from the five most common bean origins.
choc["country_of_bean_origin"].value_counts(normalize=True).head(5).round(3)

country_of_bean_origin
Venezuela             0.100
Peru                  0.096
Dominican Republic    0.089
Ecuador               0.087
Madagascar            0.070
Name: proportion, dtype: float64

In [5]:
# A unique ID would have as many distinct values as there are rows.
print("Distinct ref values:", choc["ref"].nunique())
print("Rows:", len(choc))

Distinct ref values: 630
Rows: 2530


**Answer.**

- **Coverage:** 2,530 bars from 580 makers, using beans from 62 countries, reviewed between 2006 and 2021.
- **Top origins:** Venezuela (10.0%), Peru (9.6%), the Dominican Republic (8.9%), Ecuador (8.7%) and Madagascar (7.0%). No single origin dominates: the top five together account for under half the bars.
- **An ID?** `ref` is the obvious candidate, but it is **not unique**. It has only 630 distinct values for 2,530 rows, because it numbers the *review batch*: bars reviewed at the same time share a `ref`. No column identifies a single bar, so a bar is really identified by the combination of maker, bean origin/bar name, and review year. Never assume a column called "ref" or "id" is unique; compare `nunique()` with the number of rows.

## Exercise 2 — Can we trust it? *(~20 min)*

**Exercise 2.** Decide what in this data needs validating or cleaning, and clean it. For each decision, state what you changed or left alone, and why.

Cleaning could include missing values, type conversions, invalid or impossible values, duplicates, and typos or inconsistent labels.

Also, from the `ingredients` column:

- A function `count_ingredients(text)` that returns the number of ingredients in a bar (`"3- B,S,C"` → `3`), used to add a column `num_ingredients`.
- A function `has_ingredient(text, code)` that says whether a bar contains a given ingredient, used to report how many bars contain each ingredient in this dictionary:

```python
codes = {"B": "beans", "S": "sugar", "S*": "other sweetener", "C": "cocoa butter",
         "V": "vanilla", "L": "lecithin", "Sa": "salt"}
```

In [6]:
# Column types: look for numbers stored as text.
choc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2530 entries, 0 to 2529
Data columns (total 10 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   ref                               2530 non-null   int64  
 1   company_manufacturer              2530 non-null   object 
 2   company_location                  2530 non-null   object 
 3   review_date                       2530 non-null   int64  
 4   country_of_bean_origin            2530 non-null   object 
 5   specific_bean_origin_or_bar_name  2530 non-null   object 
 6   cocoa_percent                     2530 non-null   object 
 7   ingredients                       2443 non-null   object 
 8   most_memorable_characteristics    2530 non-null   object 
 9   rating                            2530 non-null   float64
dtypes: float64(1), int64(2), object(7)
memory usage: 197.8+ KB


In [7]:
# One line per text column: how many distinct values, how many missing.
text_cols = ["company_manufacturer", "company_location", "country_of_bean_origin",
             "specific_bean_origin_or_bar_name", "cocoa_percent", "ingredients",
             "most_memorable_characteristics"]

for col in text_cols:
    print(f"{col}: {choc[col].nunique()} unique, {choc[col].isna().sum()} missing")

company_manufacturer: 580 unique, 0 missing
company_location: 67 unique, 0 missing
country_of_bean_origin: 62 unique, 0 missing
specific_bean_origin_or_bar_name: 1605 unique, 0 missing
cocoa_percent: 46 unique, 0 missing
ingredients: 21 unique, 87 missing
most_memorable_characteristics: 2487 unique, 0 missing


In [8]:
# Validate the numeric columns: are ratings and years inside their allowed ranges? Any duplicate rows?
print("Ratings from", choc["rating"].min(), "to", choc["rating"].max())
print("Years from", choc["review_date"].min(), "to", choc["review_date"].max())
print("Duplicate rows:", choc.duplicated().sum())

Ratings from 1.0 to 4.0
Years from 2006 to 2021
Duplicate rows: 0


In [9]:
# Remove the % sign, then convert the text to a number.
choc["cocoa_percent"] = choc["cocoa_percent"].str.replace("%", "").astype(float)
choc["cocoa_percent"].describe()

count    2530.000000
mean       71.639723
std         5.616724
min        42.000000
25%        70.000000
50%        70.000000
75%        74.000000
max       100.000000
Name: cocoa_percent, dtype: float64

In [10]:
# The count is everything before the "-". Missing values stay missing.
def count_ingredients(text):
    if not isinstance(text, str):   # a missing value is not text
        return pd.NA
    return int(text.split("-")[0])

choc["num_ingredients"] = choc["ingredients"].apply(count_ingredients).astype("Int64")
choc["num_ingredients"].value_counts(dropna=False).sort_index()

num_ingredients
1          6
2        750
3       1023
4        469
5        191
6          4
<NA>      87
Name: count, dtype: Int64

In [11]:
# True if a bar's ingredient list contains exactly this code.
def has_ingredient(text, code):
    if not isinstance(text, str):   # a missing value is not text
        return False
    codes_part = text.split("-")[1].replace(" ", "")   # "3- B,S,C"  ->  "B,S,C"
    return code in codes_part.split(",")      # "B,S,C"     ->  ["B", "S", "C"]

has_ingredient("3- B,S*,C", "S")

False

In [12]:
# Count the bars containing each ingredient.
codes = {"B": "beans", "S": "sugar", "S*": "other sweetener", "C": "cocoa butter",
         "V": "vanilla", "L": "lecithin", "Sa": "salt"}

for code, name in codes.items():
    count = 0
    for text in choc["ingredients"]:
        if has_ingredient(text, code):
            count = count + 1
    print(f"{name} ({code}): {count} bars")

beans (B): 2443 bars
sugar (S): 2360 bars
other sweetener (S*): 76 bars
cocoa butter (C): 1668 bars
vanilla (V): 353 bars
lecithin (L): 493 bars
salt (Sa): 37 bars


In [13]:
# All maker locations, alphabetically, to spot inconsistent names.
choc["company_location"].value_counts().sort_index()

company_location
Amsterdam      12
Argentina       9
Australia      53
Austria        30
Belgium        63
             ... 
U.S.A.       1136
Vanuatu         4
Venezuela      31
Vietnam        16
Wales           1
Name: count, Length: 67, dtype: int64

In [14]:
# Map each inconsistent location onto the country name used elsewhere.
choc["company_location"] = choc["company_location"].replace({
    "Amsterdam": "Netherlands",
    "Scotland": "U.K.",
    "Wales": "U.K.",
    "Sao Tome & Principe": "Sao Tome",
})
choc["company_location"].nunique()

63

**Answer.** A defensible pass through the file, in the order an analyst would do it:

- **What was checked and found fine.**
  - **Duplicates:** there are no duplicate rows.
  - **Ratings** all lie between 1 and 4, inside the scale.
  - **Review years** run 2006–2021, which is plausible.
  - **Missing values:** only `ingredients` has any.
- **Type conversion.** `cocoa_percent` is stored as text (`"76%"`) because of the `%` sign. Until it is converted, you cannot average it or compare it with the rating. After conversion it runs from 42% to 100%, and most bars are 70%. 100% is unusual but real: unsweetened bars exist.
- **Missing values: kept, not dropped.** 87 bars have no ingredient list. That says nothing wrong about their rating, cocoa content or origin, so dropping them would throw away 87 valid ratings. They stay in, as `<NA>`, and simply drop out of any calculation that uses `num_ingredients`. That is why the column uses the `"Int64"` type: a plain `int` column cannot hold a missing value.
- **Ingredients.** Most bars have 2 or 3 ingredients. Almost every bar has beans (2,443) and sugar (2,360). Vanilla (353), lecithin (493) and salt (37) are much rarer, which matters because many buyers treat vanilla and lecithin as a sign of a lower-quality bar.
  - **The trap:** checking `"S" in text` on the raw string also matches the "S" inside `"S*"` and `"Sa"`, and counts 2,436 bars instead of 2,360. Splitting the codes into a list and checking membership in the *list* tests for the exact code.
- **Inconsistent labels.** `company_location` mixes levels:
  - `Amsterdam` is a city, not a country.
  - `Scotland` and `Wales` are parts of the U.K., which already appears.
  - `Sao Tome` and `Sao Tome & Principe` are the same country.

  After mapping there are 63 locations instead of 67. Each mapping is a judgement call, so write it down. Someone comparing Scottish and English makers would want to keep `Scotland`.
- **Left alone.** `most_memorable_characteristics` is free text with 2,487 distinct values. It is useful for reading, but there is nothing to standardize without a much bigger text-analysis effort.

## Exercise 3 — Which bars are worth stocking? *(~20 min)*

**Exercise 3.** Find out which bars rate best.

The Flavors of Cacao scale labels ratings as follows:

| Rating | Label |
| --- | --- |
| 1.0 – 1.9 | Unpleasant |
| 2.0 – 2.9 | Disappointing |
| 3.0 – 3.49 | Recommended |
| 3.5 – 3.9 | Highly recommended |
| 4.0 – 5.0 | Outstanding |

1. Add a column with the rating label for each bar.
2. Which bean-origin countries produce the best bars? Keep countries with at least 50 reviews and sort from best to worst.
3. Show the number of ingredients against the rating label, as shares of each row.
4. Does a higher cocoa percentage go with a better rating?

In [15]:
# Translate a numeric rating into its label.
def rating_label(r):
    if r < 2:
        return "Unpleasant"
    elif r < 3:
        return "Disappointing"
    elif r < 3.5:
        return "Recommended"
    elif r < 4:
        return "Highly recommended"
    else:
        return "Outstanding"

# An ordered category keeps the labels worst to best instead of alphabetical.
labels = ["Unpleasant", "Disappointing", "Recommended", "Highly recommended", "Outstanding"]
choc["label"] = pd.Categorical(choc["rating"].apply(rating_label), categories=labels, ordered=True)
choc["label"].value_counts(sort=False)

label
Unpleasant             17
Disappointing         549
Recommended           987
Highly recommended    865
Outstanding           112
Name: count, dtype: int64

In [16]:
# Reviews and average rating per bean origin, for origins with enough reviews to trust.
by_origin = choc.groupby("country_of_bean_origin").agg(
    reviews=("rating", "count"),
    avg_rating=("rating", "mean"),
)
by_origin[by_origin["reviews"] >= 50].sort_values("avg_rating", ascending=False).round(2)

,reviews,avg_rating
country_of_bean_origin,,
Vietnam,73,3.29
Papua New Guinea,50,3.28
Madagascar,177,3.27
Brazil,78,3.26
Guatemala,62,3.26
Nicaragua,100,3.26
Tanzania,79,3.23
Belize,76,3.23
Venezuela,253,3.23


In [17]:
# Count the bars for each ingredient count and label...
table = pd.crosstab(choc["num_ingredients"], choc["label"])

# ...then divide each column by the number of bars with that ingredient count.
bars = choc["num_ingredients"].value_counts()
for col in labels:
    table[col] = table[col] / bars
table.round(2)

label,Unpleasant,Disappointing,Recommended,Highly recommended,Outstanding
num_ingredients,,,,,
1,0.0,0.33,0.5,0.17,0.0
2,0.0,0.21,0.39,0.36,0.03
3,0.0,0.17,0.4,0.37,0.06
4,0.0,0.27,0.39,0.3,0.03
5,0.01,0.28,0.39,0.26,0.06
6,0.0,0.75,0.0,0.25,0.0


In [18]:
# Average rating within cocoa ranges.
def cocoa_range(p):
    if p <= 65:
        return "65% or less"
    elif p <= 70:
        return "66-70%"
    elif p <= 75:
        return "71-75%"
    elif p <= 80:
        return "76-80%"
    else:
        return "over 80%"

choc["cocoa_range"] = choc["cocoa_percent"].apply(cocoa_range)
choc.groupby("cocoa_range")["rating"].agg(["count", "mean"]).round(2)

,count,mean
cocoa_range,,
65% or less,239,3.11
66-70%,1193,3.27
71-75%,789,3.18
76-80%,189,3.11
over 80%,120,2.85


**Answer.**

1. **Rating labels.** Only 112 bars (4%) are Outstanding, and only 17 are Unpleasant. Most bars are Recommended (987) or Highly recommended (865).
2. **Best origins.** Vietnam (3.29), Papua New Guinea (3.28) and Madagascar (3.27) top the list. But the whole range among well-reviewed origins is only 3.04 to 3.29, about one step on a scale that moves in 0.25 steps, so origin alone is a weak buying rule.
   - The clearest signal is at the bottom: **"Blend"** bars (beans mixed from several origins) score lowest, at 3.04.
   - The minimum of 50 reviews matters. Without it, an origin with two lucky bars would top the table.
3. **Ingredients.** Bars with 2–3 ingredients are the most likely to be Highly recommended or Outstanding (39–43%). Bars with 4–5 ingredients drop to 32–33%. The rows for 1 and 6 ingredients have only 6 and 4 bars, too few to read anything into.
4. **Cocoa.** Not simply. Ratings *rise* up to 66–70% cocoa (3.27), then *fall* steadily, down to 2.85 above 80%. More cocoa helps only up to about 70%; beyond that it hurts. A one-line summary such as "more cocoa, better bar" would miss this rise-then-fall shape.

**Recommendation:** single-origin bars around 70% cocoa, with few ingredients, are the safest picks.

# Part B — Hotel bookings

This dataset holds bookings for two hotels in Portugal, a **City Hotel** in Lisbon and a **Resort Hotel** in the Algarve. A cancelled booking is a room that might have been sold to someone else. **Goal:** find out who cancels, when, and what the hotels can do about it.

Each row is one booking that arrived (or was due to arrive) between July 2015 and August 2017. The columns you will use most:

| Column | Meaning |
| --- | --- |
| `hotel` | City Hotel or Resort Hotel |
| `is_canceled` | 1 if the booking was cancelled, 0 if not |
| `lead_time` | Days between making the booking and the arrival date |
| `arrival_date_year`, `arrival_date_month`, `arrival_date_day_of_month` | Arrival date, split into three columns |
| `stays_in_weekend_nights`, `stays_in_week_nights` | Nights booked, split into weekend and weekday nights |
| `adults`, `children`, `babies` | Number of guests |
| `meal` | Meal plan: BB = bed & breakfast, HB = half board, FB = full board, SC or Undefined = no meals |
| `market_segment` | How the booking was sold (Online travel agent, Groups, Direct, …) |
| `deposit_type` | No Deposit, Refundable, or Non Refund |
| `agent`, `company` | ID of the travel agent or company that made the booking, if any |
| `adr` | Average daily rate: the price per night, in euros |

The full data dictionary, covering all 32 columns, is in the [dataset's documentation](https://github.com/rfordatascience/tidytuesday/tree/main/data/2020/2020-02-11).

In [19]:
# Load the bookings. This file is larger (about 17 MB), so it takes a few seconds.
hotels = pd.read_csv("https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2020/2020-02-11/hotels.csv")
print(hotels.shape)
hotels.head()

(119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


## Exercise 4 — What are we working with? *(~20 min)*

**Exercise 4.** Describe what the data covers.

1. How many bookings are there, and which years do they span?
2. How are the bookings split between the two hotels?
3. What share of bookings is cancelled, overall and at each hotel?

In [20]:
# Size of the table and the years it covers.
print(hotels.shape)
print("Arrival years:", hotels["arrival_date_year"].min(), "to", hotels["arrival_date_year"].max())

(119390, 32)
Arrival years: 2015 to 2017


In [21]:
# Share of bookings at each hotel.
hotels["hotel"].value_counts(normalize=True).round(3)

hotel
City Hotel      0.664
Resort Hotel    0.336
Name: proportion, dtype: float64

In [22]:
# The mean of a 0/1 column is the share of 1s.
cancel_rate = hotels["is_canceled"].mean()
print(f"Cancellation rate: {round(cancel_rate * 100, 1)}%")

Cancellation rate: 37.0%


In [23]:
# Cancellation rate for each hotel, one hotel at a time.
rates = {}
for name in hotels["hotel"].unique():
    rates[name] = hotels.loc[hotels["hotel"] == name, "is_canceled"].mean()

for name, rate in rates.items():
    print(f"{name}: {round(rate * 100, 1)}%")

Resort Hotel: 27.8%
City Hotel: 41.7%


In [24]:
# The same answer in one line.
hotels.groupby("hotel")["is_canceled"].mean()

hotel
City Hotel      0.417270
Resort Hotel    0.277634
Name: is_canceled, dtype: float64

**Answer.**

- **Coverage:** 119,390 bookings for arrivals in 2015–2017. Exercise 5 shows the exact range is July 2015 to August 2017, so the first and last years are partial.
- **Split:** two thirds of the bookings (66.4%) are for the City Hotel, and one third (33.6%) for the Resort.
- **Cancellations:** **37.0%** of all bookings were cancelled, more than one in three. The City Hotel cancels far more often (41.7%) than the Resort (27.8%).

The loop and the `groupby` give the same numbers: `groupby` is the loop written for you. Either is a fine way to answer the question.

## Exercise 5 — Can we trust it? *(~20 min)*

**Exercise 5.** Decide what in this data needs validating or cleaning, and clean it. For each decision, state what you changed or left alone, and why.

Cleaning could include missing values, type conversions, invalid or impossible values, duplicates, and typos or inconsistent labels.

Also add:

- The arrival date of each booking as a real date, and its day of the week.
- The number of nights booked and the revenue of each booking (a cancelled booking brings in nothing).
- The season of each booking: Winter, Spring, Summer or Fall.

In [25]:
# Missing values, only for the columns that have any.
missing = hotels.isna().sum()
missing[missing > 0]

children         4
country        488
agent        16340
company     112593
dtype: int64

In [26]:
# How many rows exactly repeat an earlier row?
print("Duplicate rows:", hotels.duplicated().sum())

Duplicate rows: 31994


In [27]:
# Who are the duplicated rows, and how often were they cancelled?
dupes = hotels[hotels.duplicated(keep=False)]
print(dupes["market_segment"].value_counts(normalize=True).round(2).head(4))
print("Cancellation rate among duplicated rows:", round(dupes["is_canceled"].mean(), 2))
print("Cancellation rate if duplicates are dropped:", round(hotels[~hotels.duplicated()]["is_canceled"].mean(), 3))

market_segment
Groups           0.42
Offline TA/TO    0.30
Online TA        0.21
Corporate        0.04
Name: proportion, dtype: float64
Cancellation rate among duplicated rows: 0.58


Cancellation rate if duplicates are dropped: 0.275


In [28]:
# The extremes of the nightly price.
hotels["adr"].describe()

count    119390.000000
mean        101.831122
std          50.535790
min          -6.380000
25%          69.290000
50%          94.575000
75%         126.000000
max        5400.000000
Name: adr, dtype: float64

In [29]:
# Inspect the two extreme rows before deciding anything about them.
cols = ["hotel", "adr", "adults", "market_segment"]
print(hotels.loc[hotels["adr"] < 0, cols])
print(hotels.loc[hotels["adr"] > 1000, cols])

              hotel   adr  adults market_segment
14969  Resort Hotel -6.38       2         Groups
            hotel     adr  adults market_segment
48515  City Hotel  5400.0       2  Offline TA/TO


In [30]:
# A negative price and a 5,400-euro room for two are impossible: mark them missing.
hotels.loc[hotels["adr"] < 0, "adr"] = pd.NA
hotels.loc[hotels["adr"] > 1000, "adr"] = pd.NA

# Where do the zero-price bookings come from?
hotels.loc[hotels["adr"] == 0, "market_segment"].value_counts()

market_segment
Complementary    680
Online TA        367
Offline TA/TO    332
Groups           252
Direct           240
Corporate         82
Aviation           6
Name: count, dtype: int64

In [31]:
# Keep only bookings with at least one guest.
guests = hotels["adults"] + hotels["children"].fillna(0) + hotels["babies"]
print("Bookings with no guests:", (guests == 0).sum())
hotels = hotels[guests > 0]
hotels.shape

Bookings with no guests: 180


(119210, 32)

In [32]:
# Same meal plan, two spellings: use one.
hotels["meal"] = hotels["meal"].replace({"Undefined": "SC"})

# Missing children -> 0 guests; missing agent/company -> 0, meaning "none".
hotels["children"] = hotels["children"].fillna(0).astype("Int64")
hotels["agent"] = hotels["agent"].fillna(0).astype(int)
hotels["company"] = hotels["company"].fillna(0).astype(int)

hotels["meal"].value_counts()

meal
BB    92236
HB    14458
SC    11718
FB      798
Name: count, dtype: int64

In [33]:
# Arrival date: glue the three parts into text like "1 July 2015", then convert it to a date.
date_text = (hotels["arrival_date_day_of_month"].astype(str) + " "
             + hotels["arrival_date_month"] + " "
             + hotels["arrival_date_year"].astype(str))
hotels["arrival_date"] = pd.to_datetime(date_text)
hotels["arrival_day"] = hotels["arrival_date"].dt.day_name()

print("Arrivals from", hotels["arrival_date"].min().date(), "to", hotels["arrival_date"].max().date())
hotels["arrival_day"].value_counts()

Arrivals from 2015-07-01 to 2017-08-31


arrival_day
Friday       19602
Thursday     19225
Monday       18152
Saturday     18036
Wednesday    16099
Sunday       14125
Tuesday      13971
Name: count, dtype: int64

In [34]:
# Nights booked, and the money each booking actually brought in.
hotels["total_nights"] = hotels["stays_in_weekend_nights"] + hotels["stays_in_week_nights"]
hotels["revenue"] = hotels["adr"] * hotels["total_nights"]
hotels.loc[hotels["is_canceled"] == 1, "revenue"] = 0

hotels.groupby("hotel")["revenue"].sum()

hotel
City Hotel      14385342.00
Resort Hotel    11601634.03
Name: revenue, dtype: float64

In [35]:
# Map each month to its season.
def season(month):
    if month in ["December", "January", "February"]:
        return "Winter"
    elif month in ["March", "April", "May"]:
        return "Spring"
    elif month in ["June", "July", "August"]:
        return "Summer"
    else:
        return "Fall"

hotels["season"] = hotels["arrival_date_month"].apply(season)
hotels["season"].value_counts()

season
Summer    37434
Spring    32626
Fall      28418
Winter    20732
Name: count, dtype: int64

**Answer.** A defensible pass through the file:

- **Missing values: four columns, three different meanings.**
  - `children` (4 missing) becomes 0, a safe assumption for 4 rows out of 119,390.
  - `country` (488 missing) is genuinely unknown and is left as missing. Nothing in this analysis depends on it.
  - `agent` (16,340) and `company` (112,593) are different: a booking with no travel agent, or not made through a company, has nothing to record. Missing here means **"none"**, not "unknown", so they are filled with 0. Dropping rows with a missing `company` would throw away 94% of the data.
  - Note that `agent` and `company` hold *ID numbers*, not quantities. An average agent ID means nothing.
- **Duplicates: kept, with a caveat.** 31,994 rows (27%) are exact copies of another row. The file has no booking ID, so there is no way to tell a true duplicate (the same booking entered twice) from two genuinely separate bookings that happen to be identical.
  - Identical bookings are common: a tour group books ten identical rooms on the same day, and 42% of the duplicated rows are Groups bookings.
  - The decision matters: dropping them moves the headline cancellation rate from 37.0% to 27.5%.
  - There is no single right answer. We **keep them**, because they are plausibly real rooms, but any number we report carries the caveat.
- **Impossible prices: set to missing, not deleted.**
  - One booking has a **negative** price (−6.38), and one City Hotel room for two adults cost **5,400 euros** a night. Both are data-entry errors.
  - The rest of each booking is still valid, so only the price is removed.
- **Zero prices: left alone.** The 1,959 zero-price bookings are a different case.
  - The largest group is `Complementary` (680): rooms given away on purpose. Those are correct, not errors.
  - The rest could be staff stays, vouchers or errors, and there is no way to tell. A "zero" is not automatically a missing value.
- **Impossible bookings: removed.** 180 bookings have no guests at all. They cannot be real stays, so they are removed, leaving 119,210 bookings.
- **Inconsistent labels:** according to the column table above, the meal plans `Undefined` and `SC` both mean "no meals". All 11,718 bookings without meals now carry one label.
- **Type conversion:** `children` becomes a whole-number column (`"Int64"`).
- **Arrival date:** arrivals run from 1 July 2015 to 31 August 2017. Friday and Thursday are the busiest arrival days, and Tuesday and Sunday the quietest.
- **Nights and revenue:** bookings that were not cancelled brought in about 14.4 million euros at the City Hotel and 11.6 million at the Resort.
- **Season:** Summer is the busiest season (37,434 bookings) and Winter the quietest (20,732).

## Exercise 6 — Who cancels? *(~20 min)*

**Exercise 6.** Find out which bookings get cancelled.

1. Does the kind of stay matter? Compare weekend-only, weekday-only and mixed stays.
2. Which sales channels cancel the most, and how do they compare on volume and price?
3. Are bookings made further ahead more likely to be cancelled?
4. Compare cancellation rates across deposit types. Is anything surprising, and is it believable?

In [36]:
# Classify a booking by which kinds of nights it includes.
def stay_type(row):
    if row["stays_in_weekend_nights"] > 0 and row["stays_in_week_nights"] > 0:
        return "Mixed"
    elif row["stays_in_weekend_nights"] > 0:
        return "Weekend only"
    elif row["stays_in_week_nights"] > 0:
        return "Weekday only"
    else:
        return "No nights"

hotels["stay_type"] = hotels.apply(stay_type, axis=1)
hotels.groupby("stay_type")["is_canceled"].agg(["count", "mean"]).round(3)

,count,mean
stay_type,,
Mixed,60388,0.378
No nights,645,0.036
Weekday only,51250,0.380
Weekend only,6927,0.271


In [37]:
# Volume, cancellation rate and price for each sales channel.
by_segment = hotels.groupby("market_segment").agg(
    bookings=("is_canceled", "count"),
    cancel_rate=("is_canceled", "mean"),
    avg_price=("adr", "mean"),
)
by_segment.sort_values("cancel_rate", ascending=False).round(2)

,bookings,cancel_rate,avg_price
market_segment,,,
Undefined,2,1.00,15.00
Groups,19791,0.61,79.56
Online TA,56408,0.37,117.32
Offline TA/TO,24182,0.34,87.26
Aviation,235,0.22,100.99
Corporate,5282,0.19,69.53
Direct,12582,0.15,115.63
Complementary,728,0.12,2.95


In [38]:
# Group bookings by how far ahead they were made.
def lead_band(days):
    if days <= 7:
        return "0-7 days"
    elif days <= 30:
        return "8-30 days"
    elif days <= 90:
        return "31-90 days"
    elif days <= 180:
        return "91-180 days"
    elif days <= 365:
        return "181-365 days"
    else:
        return "over a year"

# An ordered category lists the bands shortest to longest instead of alphabetically.
band_order = ["0-7 days", "8-30 days", "31-90 days", "91-180 days", "181-365 days", "over a year"]
hotels["lead_band"] = pd.Categorical(hotels["lead_time"].apply(lead_band), categories=band_order, ordered=True)
hotels.groupby("lead_band", observed=False)["is_canceled"].agg(["count", "mean"]).round(3)

,count,mean
lead_band,,
0-7 days,19635,0.096
8-30 days,18945,0.279
31-90 days,29529,0.377
91-180 days,26420,0.447
181-365 days,21534,0.555
over a year,3147,0.677


In [39]:
# Number of bookings and cancellation rate for each deposit type.
hotels.groupby("deposit_type")["is_canceled"].agg(["count", "mean"]).round(3)

,count,mean
deposit_type,,
No Deposit,104461,0.284
Non Refund,14587,0.994
Refundable,162,0.222


In [40]:
# Look closer at the Non Refund bookings.
non_refund = hotels[hotels["deposit_type"] == "Non Refund"]
print("Non Refund bookings:", len(non_refund))
print("...of which exact duplicates:", non_refund.duplicated().sum())
print(non_refund["market_segment"].value_counts().head(3))
print("Median lead time, Non Refund:", non_refund["lead_time"].median(), "days; all bookings:", hotels["lead_time"].median(), "days")

Non Refund bookings: 14587
...of which exact duplicates: 13549
market_segment
Groups           9172
Offline TA/TO    5006
Corporate         334
Name: count, dtype: int64
Median lead time, Non Refund: 183.0 days; all bookings: 69.0 days


**Answer.**

1. **Stay type.** Bookings that include weekday nights cancel at about 38%, whether or not they also include a weekend. Weekend-only stays cancel less (27%). The 645 "No nights" bookings (same-day use of a room) almost never cancel (4%).
2. **Sales channels.** Ignore `Undefined`: 2 bookings tell you nothing.
   - **Groups** cancel the most by far (61%), followed by online travel agents (37%).
   - **Direct** (15%) and **Corporate** (19%) bookings are the most reliable.
   - Online travel agents bring the most bookings (56,408) at the highest prices (117 euros). They are the channel the hotels can least afford to lose, and also one that cancels often.
3. **Booking ahead.** The further ahead a booking is made, the more likely it is to be cancelled. The rate climbs steadily from **10%** for bookings made within a week to **68%** for bookings made over a year ahead.
   - This is the clearest pattern in the data, and the most useful one for the hotels.
   - Lead time is known at the moment of booking, so it can drive a policy, for example a deposit for bookings made more than 90 days out.
4. **Deposits — the surprise:** bookings with a *Non Refund* deposit, where the guest loses their money if they cancel, are cancelled **99.4%** of the time. Bookings with no deposit at all are cancelled 28.4% of the time. That is the opposite of what a deposit is for.
   - **A closer look suggests the problem is in the data, not the guests.** 13,549 of the 14,587 Non Refund bookings are exact duplicates of another row. They are mostly Groups and offline travel-agent bookings, made far in advance (median 183 days vs 69). This looks like blocks of rooms held by tour operators, released in bulk and recorded as "cancelled", rather than individual guests walking away from their money.
   - **The lesson:** when a result contradicts common sense, check the data before you build a recommendation on it. "Introduce non-refundable deposits to cut cancellations" would be the wrong conclusion from this table.

## Exercise 7 — Seasonality and what drives cancellations *(~20 min)*

**Exercise 7.** Look at prices over the year and at what goes with cancellation.

1. Show the average nightly price by month for each hotel, in calendar order, with overall averages. Which seasons are priced above the year's average?
2. Produce the same monthly averages as a long table, with one row per month and hotel.
3. How do cancelled bookings differ from bookings that were kept? Which of those differences are trustworthy?

In [41]:
# Calendar order for the months, so tables sort January to December instead of alphabetically.
months = ["January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]
hotels["arrival_date_month"] = pd.Categorical(hotels["arrival_date_month"], categories=months, ordered=True)

In [42]:
# Average nightly price by month and hotel, with overall averages.
price_table = pd.pivot_table(hotels, values="adr", index="arrival_date_month", columns="hotel",
                             aggfunc="mean", margins=True, observed=False).round(2)
price_table

hotel,City Hotel,Resort Hotel,All
arrival_date_month,,,
January,82.75,49.51,70.45
February,85.33,55.19,73.72
March,92.15,57.57,80.35
April,111.40,77.85,100.47
May,121.76,78.76,108.77
June,119.19,110.48,116.76
July,110.95,155.18,126.94
August,114.86,186.79,140.26
September,110.12,93.25,105.13


In [43]:
# Flag the seasons that are pricier than the year's average.
overall_price = hotels["adr"].mean()
print("Overall average price:", round(overall_price, 2))

for s in ["Winter", "Spring", "Summer", "Fall"]:
    season_price = hotels.loc[hotels["season"] == s, "adr"].mean()
    if season_price > overall_price:
        print(s, round(season_price, 2), "PEAK")
    else:
        print(s, round(season_price, 2))

Overall average price: 101.93
Winter 75.26
Spring 97.44
Summer 128.9 PEAK
Fall 90.99


In [44]:
# Grouping by two columns gives one row per month and hotel.
price_long = hotels.groupby(["arrival_date_month", "hotel"], as_index=False, observed=False).agg(
    avg_price=("adr", "mean"),
).round(2)
print(price_long.shape)
price_long.head()

(24, 3)


,arrival_date_month,hotel,avg_price
0,January,City Hotel,82.75
1,January,Resort Hotel,49.51
2,February,City Hotel,85.33
3,February,Resort Hotel,55.19
4,March,City Hotel,92.15


In [45]:
# Average of each booking characteristic, for kept (0) and cancelled (1) bookings.
compare = ["lead_time", "previous_cancellations", "booking_changes", "total_of_special_requests",
           "required_car_parking_spaces", "adr", "total_nights", "revenue", "agent"]
hotels.groupby("is_canceled")[compare].mean().round(2)

,lead_time,previous_cancellations,booking_changes,total_of_special_requests,required_car_parking_spaces,adr,total_nights,revenue,agent
is_canceled,,,,,,,,,
0,80.08,0.02,0.29,0.71,0.1,100.17,3.39,346.45,78.74
1,144.89,0.21,0.10,0.33,0.0,104.90,3.49,0.00,68.36


**Answer.**

1. **Prices by month.** The two hotels have opposite seasons.
   - The **Resort** almost quadruples its price from January (49.51) to August (186.79). It is a summer beach business.
   - The **City Hotel** is much flatter (83–122), and peaks in spring rather than summer, when city travel and conferences pick up.
   - Only **Summer** (128.90 euros) is priced above the year's average (101.93). Winter is the cheapest (75.26).
   - The table needs the months as an ordered category. Otherwise it lists them alphabetically, starting with April.
2. **Long format.** The `groupby` table has 24 rows: 12 months × 2 hotels, with the same averages as the pivot table. The two tables hold the same numbers in different shapes. The wide pivot table is easier for a person to read, and the long table is the shape a database or another program expects. Grouping by two columns gives the long shape directly.
3. **Cancelled vs kept bookings.** Read the table with suspicion.
   - **`revenue` (0 vs 346 euros) shows the biggest difference, and it is fake.** We set revenue to 0 for every cancelled booking in Exercise 5, so it is built from the answer. A column made *from* the outcome will always look like it explains the outcome.
   - **`agent` is an ID number.** Agent 240 is not "more" than agent 9, so an average agent ID (79 vs 68) means nothing.
   - **The real signals:**
     - **Lead time:** cancelled bookings were made almost twice as far ahead (145 vs 80 days), which matches Exercise 6.
     - **Special requests and parking:** kept bookings ask for more special requests (0.71 vs 0.33) and parking spaces (0.10 vs almost none). A guest who asks for a cot or a parking space is planning to come.
     - **Past behaviour:** guests who cancelled before cancel again (0.21 vs 0.02 previous cancellations).
   - **Booking changes, with care:** kept bookings were changed about three times as often (0.29 vs 0.10 changes). Changes are counted until check-in *or cancellation*, so a booking cancelled early had less time to be changed, and a guest who changes their dates is already committed to coming. This one is only known after the fact, so it cannot drive a policy at the moment of booking the way lead time can.
   - Price (105 vs 100 euros) and the number of nights booked (3.5 vs 3.4) barely differ. A cancelled booking still has nights recorded: they are the nights the guest *booked*, not nights they stayed.

   These are associations, not causes. Asking for a parking space does not make a guest show up. It reveals a guest who already intends to.

---

### Data sources

- **Chocolate bar ratings** — Brady Brelinski, [Flavors of Cacao](http://flavorsofcacao.com/chocolate_database.html), via [TidyTuesday, 18 January 2022](https://github.com/rfordatascience/tidytuesday/tree/main/data/2022/2022-01-18).
- **Hotel bookings** — Nuno Antonio, Ana de Almeida and Luis Nunes (2019), "Hotel booking demand datasets", *Data in Brief* 22, 41–49, [doi:10.1016/j.dib.2018.11.126](https://doi.org/10.1016/j.dib.2018.11.126), via [TidyTuesday, 11 February 2020](https://github.com/rfordatascience/tidytuesday/tree/main/data/2020/2020-02-11).